# Test to start server launcher for multi-agents using llama-agents

key-features of agent in llama-agents of llama-index
- each agent has its own running microservice
- control plane
  - routes and distributes tasks via customizable LLM
- message queue
  - pass messagess between agents using a message queue
- agentic orchestrator
  - decides which agents are relevant to the task

compoments
- control pane server
- message queue
  - single message queue
- agent service
- launcher
  - local launcher
  - server launcer
- function tool
- function calling agent worker
- callback message consumer

launching
- message queue server
- control plane server
- ```tool agent``` servers

## multi-agent system

In [1]:
import dotenv
dotenv.load_dotenv() # our .env file defines OPENAI_API_KEY

from llama_agents import (
    AgentOrchestrator,
    AgentService,
    CallableMessageConsumer,
    ControlPlaneServer,
    ServerLauncher,
    SimpleMessageQueue,
)
from llama_index.core.agent import FunctionCallingAgentWorker
from llama_index.core.tools import FunctionTool
from llama_index.llms.ollama import Ollama
from llama_index.llms.openai import OpenAI


import logging

# turn on logging so we can see the system working
logging.getLogger("llama_agents").setLevel(logging.INFO)

## LLM ready

In [2]:
# ResponseError: tiger-gemma2 does not support tools
# llm = Ollama(
#     model='tiger-gemma2',
#     request_timeout=30000.0,
#     keep_alive="10m",
#     additional_kwargs={"mirostat": 0, "keep_alive": "10m"}
# )

import os
llm = OpenAI()

## set single message queue and control plane

In [3]:
# Set up the message queue and control plane
message_queue = SimpleMessageQueue()

control_plane = ControlPlaneServer(
    message_queue=message_queue,
    orchestrator=AgentOrchestrator(llm=llm),
    port=37000
)

## set agents for tools

In [4]:
# create a tool
def get_the_secret_fact() -> str:
    """Returns the secret fact."""
    return "The secret fact is: A baby llama is called a 'Cria'."

tool = FunctionTool.from_defaults(fn=get_the_secret_fact)

# create our agents
worker1 = FunctionCallingAgentWorker.from_tools([tool], llm=llm)
worker2 = FunctionCallingAgentWorker.from_tools([], llm=llm)
agent1 = worker1.as_agent()
agent2 = worker2.as_agent()


agent_server_1 = AgentService(
    agent=agent1,
    message_queue=message_queue,
    description="Useful for getting the secret fact.",
    service_name="secret_fact_agent",
    host="localhost",
    port=38003
)
agent_server_2 = AgentService(
    agent=agent2,
    message_queue=message_queue,
    description="Useful for getting random dumb facts.",
    service_name="dumb_fact_agent",
    host="localhost",
    port=38004
)

## start server launcher

In [ ]:
# Additional human consumer
def handle_result(message) -> None:
    print(f"Got result:", message.data)


# the final result is published to a "human" consumer
# so we define one to handle it!
human_consumer = CallableMessageConsumer(
    handler=handle_result, message_type="human"
)

# Define Launcher
launcher = ServerLauncher(
    [agent_server_1, agent_server_2],
    control_plane,
    message_queue,
    additional_consumers=[human_consumer],
)

result = await launcher.alaunch_servers()
print(result)

INFO:llama_agents.message_queues.simple - Launching message queue server at 127.0.0.1:8001
INFO:     Started server process [357264]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)
INFO:llama_agents.control_plane.server - Launching control plane server at 127.0.0.1:37000
INFO:     Started server process [357264]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:37000 (Press CTRL+C to quit)
INFO:llama_agents.message_queues.simple - Consumer ControlPlaneServer-502f64ff-55ee-4527-8cbd-c77859dd7dd0: control_plane has been registered.
INFO:llama_agents.message_queues.simple - Consumer AgentService-4f04902f-2cfa-4764-8fc1-ce0578c97301: secret_fact_agent has been registered.
INFO:llama_agents.services.agent - Launching secret_fact_agent server at localhost:38003
INFO:     Started server process [357264]
INFO

INFO:     127.0.0.1:38054 - "POST /services/register HTTP/1.1" 200 OK


INFO:llama_agents.message_queues.simple - Consumer AgentService-6ad1b038-d457-4618-a9bf-ae41237cd112: dumb_fact_agent has been registered.
INFO:llama_agents.services.agent - Launching dumb_fact_agent server at localhost:38004
INFO:     Started server process [357264]
INFO:     Waiting for application startup.
INFO:llama_agents.services.agent - Processing initiated.
INFO:     Application startup complete.


INFO:     127.0.0.1:38060 - "POST /services/register HTTP/1.1" 200 OK


INFO:     Uvicorn running on http://localhost:38004 (Press CTRL+C to quit)
INFO:llama_agents.message_queues.simple - Consumer fae56c92-99f8-4b97-b23f-1b8e59e772b6: human has been registered.
